# 🧬 Dark Manifold V28: Full Cell Cycle Simulation

**Simulate an entire E. coli division cycle (20 minutes)**

The model learns to predict:
- Metabolite dynamics throughout the cell cycle
- Cell growth and volume changes
- DNA replication progress
- Pathway activities (glycolysis, TCA, etc.)
- Division timing

Output: JSON trajectory you can visualize as a "virtual cell"

In [ ]:
#@title 1️⃣ Setup
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import pearsonr
import json

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# Cell cycle parameters (E. coli)
DOUBLING_TIME = 20 * 60  # 20 minutes in seconds
DT = 10.0  # 10 second timesteps
N_STEPS = int(DOUBLING_TIME / DT)  # 120 steps

# Training params
N_EPOCHS = 1000
BATCH_SIZE = 32
LEARNING_RATE = 1e-3

print(f'Simulating {DOUBLING_TIME/60:.0f} min cycle in {N_STEPS} steps')

In [ ]:
#@title 2️⃣ E. coli Cell Cycle Ground Truth

class EcoliCellCycle:
    '''Ground truth E. coli cell cycle dynamics.'''
    
    def __init__(self):
        self.metabolites = {
            'ATP': 3.65, 'ADP': 0.22, 'AMP': 0.06,
            'NAD': 2.6, 'NADH': 0.08, 'NADP': 0.002, 'NADPH': 0.12,
            'Glucose_ext': 40.0, 'G6P': 0.1, 'F6P': 0.06,
            'FBP': 0.02, 'G3P': 0.01, 'PEP': 0.04,
            'Pyruvate': 3.4, 'Acetyl_CoA': 0.6,
            'Citrate': 0.4, 'AKG': 0.3, 'Succinate': 0.5,
            'Malate': 0.2, 'OAA': 0.02,
            'Glutamate': 100.0, 'Glutamine': 4.0,
            'Phosphate': 17.8, 'CO2': 0.02,
        }
        self.met_names = list(self.metabolites.keys())
        self.n_met = len(self.met_names)
        self.met_idx = {m: i for i, m in enumerate(self.met_names)}
        self.cell_vars = ['Volume', 'DNA_replicated', 'Division_progress']
        self.n_cell = len(self.cell_vars)
        self.pathways = ['glucose_uptake', 'glycolysis', 'tca_cycle',
                         'atp_synthase', 'dna_replication', 'protein_synthesis',
                         'lipid_synthesis', 'cell_wall_synthesis']
        self.n_pathways = len(self.pathways)
        self.n_state = self.n_met + self.n_cell
        
    def get_initial_state(self, batch_size, variation=0.1):
        M0 = np.array([self.metabolites[m] for m in self.met_names])
        M = np.tile(M0, (batch_size, 1))
        M *= np.exp(np.random.randn(batch_size, self.n_met) * variation)
        C = np.zeros((batch_size, self.n_cell))
        C[:, 0] = 1.0
        state = np.concatenate([M, C], axis=1)
        return torch.tensor(state, dtype=torch.float32)
    
    def compute_phase(self, t, T=DOUBLING_TIME):
        frac = t / T
        if frac < 0.25: return 'G1', 0
        elif frac < 0.65: return 'S', 1
        elif frac < 0.85: return 'G2', 2
        else: return 'M', 3
    
    def step(self, state, t, dt=DT):
        batch_size = state.shape[0]
        M = state[:, :self.n_met].numpy()
        C = state[:, self.n_met:].numpy()
        phase, phase_idx = self.compute_phase(t)
        
        atp = M[:, self.met_idx['ATP']]
        adp = M[:, self.met_idx['ADP']]
        glucose = M[:, self.met_idx['Glucose_ext']]
        pyr = M[:, self.met_idx['Pyruvate']]
        energy_charge = atp / (atp + adp + 0.01)
        
        fluxes = np.zeros((batch_size, self.n_pathways))
        v_glucose = 0.5 * glucose / (0.1 + glucose)
        fluxes[:, 0] = v_glucose
        v_glyc = 0.8 * v_glucose * (1 - 0.3 * energy_charge)
        fluxes[:, 1] = v_glyc
        v_tca = 0.6 * pyr / (0.5 + pyr) * energy_charge
        fluxes[:, 2] = v_tca
        v_atp_syn = 0.9 * (1.0 - energy_charge) * (v_glyc + v_tca)
        fluxes[:, 3] = v_atp_syn
        v_dna = 0.8 * energy_charge if phase == 'S' else 0.0
        fluxes[:, 4] = v_dna
        v_prot = 0.7 * energy_charge * (1.2 if phase in ['G1', 'G2'] else 0.8)
        fluxes[:, 5] = v_prot
        v_lipid = 0.4 * energy_charge * C[:, 0]
        fluxes[:, 6] = v_lipid
        v_wall = 0.3 * energy_charge
        fluxes[:, 7] = v_wall
        
        dM = np.zeros_like(M)
        dM[:, self.met_idx['Glucose_ext']] = -v_glucose * dt
        dM[:, self.met_idx['G6P']] = v_glucose * dt - v_glyc * dt
        dM[:, self.met_idx['Pyruvate']] = 2 * v_glyc * dt - v_tca * dt
        dM[:, self.met_idx['ATP']] = (2 * v_glyc + v_atp_syn - v_prot * 4 - v_dna * 2) * dt
        dM[:, self.met_idx['ADP']] = -dM[:, self.met_idx['ATP']]
        dM[:, self.met_idx['Citrate']] = (v_tca - 0.8 * v_tca) * dt
        dM[:, self.met_idx['AKG']] = 0.8 * v_tca * dt - 0.7 * v_tca * dt
        dM[:, self.met_idx['NADH']] = (v_glyc + 3 * v_tca - v_atp_syn) * 0.1 * dt
        dM[:, self.met_idx['NAD']] = -dM[:, self.met_idx['NADH']]
        dM += np.random.randn(*dM.shape) * 0.01 * np.abs(dM)
        M_new = np.clip(M + dM, 1e-6, 1000.0)
        
        dC = np.zeros_like(C)
        growth_rate = np.log(2) / DOUBLING_TIME
        dC[:, 0] = C[:, 0] * growth_rate * dt * energy_charge
        if phase == 'S':
            dC[:, 1] = v_dna * dt / (0.4 * DOUBLING_TIME)
        C_new = C + dC
        C_new[:, 1] = np.clip(C_new[:, 1], 0, 1)
        if phase == 'M':
            C_new[:, 2] = (t - 0.85 * DOUBLING_TIME) / (0.15 * DOUBLING_TIME)
        C_new[:, 2] = np.clip(C_new[:, 2], 0, 1)
        
        next_state = np.concatenate([M_new, C_new], axis=1)
        return torch.tensor(next_state, dtype=torch.float32), torch.tensor(fluxes, dtype=torch.float32)
    
    def generate_cycle(self, batch_size=1):
        state = self.get_initial_state(batch_size)
        states = [state]
        fluxes_list = []
        phases = []
        for step in range(N_STEPS):
            t = step * DT
            state, fluxes = self.step(state, t)
            states.append(state)
            fluxes_list.append(fluxes)
            phases.append(self.compute_phase(t)[0])
        return {
            'states': torch.stack(states, dim=1),
            'fluxes': torch.stack(fluxes_list, dim=1),
            'phases': phases,
            'time': np.arange(N_STEPS + 1) * DT,
        }

ecoli = EcoliCellCycle()
test_cycle = ecoli.generate_cycle(batch_size=2)
print(f'State size: {ecoli.n_state} ({ecoli.n_met} metabolites + {ecoli.n_cell} cell vars)')
print(f'Pathways: {ecoli.n_pathways}')
print(f'Trajectory shape: {test_cycle["states"].shape}')

In [ ]:
#@title 3️⃣ V28 Cell Cycle Neural Network

class CellCycleNet(nn.Module):
    def __init__(self, n_met, n_cell, n_pathways, hidden_dim=128):
        super().__init__()
        self.n_met = n_met
        self.n_cell = n_cell
        self.n_state = n_met + n_cell
        self.n_pathways = n_pathways
        
        self.flux_net = nn.Sequential(
            nn.Linear(self.n_state, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, n_pathways),
            nn.Softplus()
        )
        self.met_update = nn.Sequential(
            nn.Linear(n_pathways + self.n_state, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, n_met)
        )
        self.cell_update = nn.Sequential(
            nn.Linear(n_pathways + self.n_state, hidden_dim // 2),
            nn.SiLU(),
            nn.Linear(hidden_dim // 2, n_cell)
        )
        self.dt_scale = nn.Parameter(torch.tensor(1.0))
        
    def step(self, state, dt=DT):
        fluxes = self.flux_net(state)
        combined = torch.cat([state, fluxes], dim=-1)
        dM = self.met_update(combined) * dt * self.dt_scale
        dC = self.cell_update(combined) * dt * self.dt_scale
        M = state[:, :self.n_met]
        C = state[:, self.n_met:]
        M_new = torch.clamp(M + dM, min=1e-6, max=1000.0)
        C_new = torch.clamp(C + dC, min=0.0, max=10.0)
        next_state = torch.cat([M_new, C_new], dim=-1)
        return next_state, fluxes
    
    def forward(self, initial_state, n_steps=N_STEPS):
        state = initial_state
        states = [state]
        fluxes_list = []
        for _ in range(n_steps):
            state, fluxes = self.step(state)
            states.append(state)
            fluxes_list.append(fluxes)
        return torch.stack(states, dim=1), torch.stack(fluxes_list, dim=1)

model = CellCycleNet(ecoli.n_met, ecoli.n_cell, ecoli.n_pathways, hidden_dim=128).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f'V28 CellCycleNet: {n_params:,} parameters')

In [ ]:
#@title 4️⃣ Training

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=N_EPOCHS)

train_losses = []
val_corrs = []
best_corr = 0
best_state = None

print('Training cell cycle model...')
for epoch in range(N_EPOCHS):
    model.train()
    gt_data = ecoli.generate_cycle(batch_size=BATCH_SIZE)
    gt_states = gt_data['states'].to(device)
    gt_fluxes = gt_data['fluxes'].to(device)
    
    initial_state = gt_states[:, 0, :]
    pred_states, pred_fluxes = model(initial_state, N_STEPS)
    
    state_loss = nn.functional.mse_loss(pred_states, gt_states)
    flux_loss = nn.functional.mse_loss(pred_fluxes, gt_fluxes)
    loss = state_loss + 0.1 * flux_loss
    
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    optimizer.step()
    scheduler.step()
    train_losses.append(loss.item())
    
    if epoch % 100 == 0 or epoch == N_EPOCHS - 1:
        model.eval()
        with torch.no_grad():
            val_data = ecoli.generate_cycle(batch_size=64)
            val_states = val_data['states'].to(device)
            pred_val, _ = model(val_states[:, 0, :], N_STEPS)
            pred_flat = pred_val.cpu().numpy().flatten()
            true_flat = val_states.cpu().numpy().flatten()
            r, _ = pearsonr(pred_flat, true_flat)
            val_corrs.append((epoch, r))
            if r > best_corr:
                best_corr = r
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            print(f'Epoch {epoch:4d} | Loss: {loss.item():.6f} | Val r: {r:.4f} | Best: {best_corr:.4f}')

print(f'Training complete. Best correlation: {best_corr:.4f}')

In [ ]:
#@title 5️⃣ Generate Virtual Cell Simulation

model.load_state_dict(best_state)
model.eval()

print('Generating virtual cell simulation...')
with torch.no_grad():
    initial = ecoli.get_initial_state(1).to(device)
    pred_states, pred_fluxes = model(initial, N_STEPS)

states = pred_states[0].cpu().numpy()
fluxes = pred_fluxes[0].cpu().numpy()
time_seconds = (np.arange(N_STEPS + 1) * DT).tolist()

metabolites = {name: states[:, i].tolist() for i, name in enumerate(ecoli.met_names)}
cell_properties = {
    'volume': states[:, ecoli.n_met].tolist(),
    'dna_replicated': states[:, ecoli.n_met + 1].tolist(),
    'division_progress': states[:, ecoli.n_met + 2].tolist(),
}
pathway_fluxes = {name: fluxes[:, i].tolist() for i, name in enumerate(ecoli.pathways)}
phases = [ecoli.compute_phase(t)[0] for t in time_seconds]

virtual_cell = {
    'metadata': {
        'organism': 'E. coli',
        'doubling_time_minutes': DOUBLING_TIME / 60,
        'timestep_seconds': DT,
        'n_steps': N_STEPS,
        'model_version': 'V28',
    },
    'time_seconds': time_seconds,
    'metabolites': metabolites,
    'cell_properties': cell_properties,
    'pathway_fluxes': pathway_fluxes,
    'phases': phases,
}

with open('virtual_cell_cycle.json', 'w') as f:
    json.dump(virtual_cell, f, indent=2)

print(f'Saved virtual_cell_cycle.json')
print(f'Duration: {DOUBLING_TIME/60:.0f} minutes ({N_STEPS} timesteps)')

In [ ]:
#@title 6️⃣ Visualize Cell Cycle

fig, axes = plt.subplots(3, 2, figsize=(14, 12))
time_min = np.array(time_seconds) / 60

ax = axes[0, 0]
for met in ['ATP', 'ADP', 'Pyruvate', 'Glucose_ext']:
    ax.plot(time_min, metabolites[met], label=met)
ax.set_xlabel('Time (min)')
ax.set_ylabel('Concentration (mM)')
ax.set_title('Key Metabolites During Cell Cycle')
ax.legend()

ax = axes[0, 1]
ax.plot(time_min, cell_properties['volume'], label='Volume', color='green')
ax.plot(time_min, cell_properties['dna_replicated'], label='DNA replicated', color='blue')
ax.plot(time_min, cell_properties['division_progress'], label='Division progress', color='red')
ax.set_xlabel('Time (min)')
ax.set_ylabel('Relative value')
ax.set_title('Cell Cycle Progression')
ax.legend()

ax = axes[1, 0]
for pathway in ['glycolysis', 'tca_cycle', 'atp_synthase']:
    ax.plot(time_min[:-1], pathway_fluxes[pathway], label=pathway)
ax.set_xlabel('Time (min)')
ax.set_ylabel('Flux')
ax.set_title('Metabolic Pathway Activity')
ax.legend()

ax = axes[1, 1]
for pathway in ['dna_replication', 'protein_synthesis', 'lipid_synthesis']:
    ax.plot(time_min[:-1], pathway_fluxes[pathway], label=pathway)
ax.set_xlabel('Time (min)')
ax.set_ylabel('Flux')
ax.set_title('Biosynthesis Activity')
ax.legend()

ax = axes[2, 0]
atp = np.array(metabolites['ATP'])
adp = np.array(metabolites['ADP'])
amp = np.array(metabolites['AMP'])
energy_charge = (atp + 0.5 * adp) / (atp + adp + amp + 0.01)
ax.plot(time_min, energy_charge, color='purple', linewidth=2)
ax.set_xlabel('Time (min)')
ax.set_ylabel('Energy Charge')
ax.set_title('Cellular Energy Status')
ax.axhline(0.85, color='green', linestyle='--', label='Optimal')
ax.legend()

ax = axes[2, 1]
phase_colors = {'G1': 'blue', 'S': 'green', 'G2': 'orange', 'M': 'red'}
for i, (t, phase) in enumerate(zip(time_min, phases)):
    ax.axvspan(t, t + DT/60, color=phase_colors.get(phase, 'gray'), alpha=0.3)
ax.set_xlabel('Time (min)')
ax.set_title('Cell Cycle Phases')
ax.set_yticks([])
for phase, color in phase_colors.items():
    ax.bar(0, 0, color=color, alpha=0.3, label=phase)
ax.legend(loc='upper right')

plt.tight_layout()
plt.savefig('cell_cycle_visualization.png', dpi=150)
plt.show()

In [ ]:
#@title 7️⃣ Save and Download

checkpoint = {
    'model_state_dict': best_state,
    'n_params': n_params,
    'best_corr': best_corr,
    'train_losses': train_losses,
    'val_corrs': val_corrs,
    'config': {
        'n_met': ecoli.n_met,
        'n_cell': ecoli.n_cell,
        'n_pathways': ecoli.n_pathways,
        'doubling_time': DOUBLING_TIME,
        'dt': DT,
        'n_steps': N_STEPS,
    },
    'metabolite_names': ecoli.met_names,
    'pathway_names': ecoli.pathways,
}

torch.save(checkpoint, 'dark_manifold_v28.pt')
print('Saved dark_manifold_v28.pt')

from google.colab import files
files.download('dark_manifold_v28.pt')
files.download('virtual_cell_cycle.json')
files.download('cell_cycle_visualization.png')